# FRACTYPE v3 — Kaggle Training Pipeline
## GOAL: Train AI to encode/decode at minimum viable density

| CP | State | Target |
|---|---|---|
| SCOOT | Single-line encode | 85 chars = 4 tiers |
| CRAWL | Cross-domain mapping | 8 projects on 1 line |
| STAND | Mycelia graph | 12 nodes x 17 hyphae |
| BOUNCE | Recursive nesting | Depth 4 corruption |
| WALK | Diegetic overflow | Corruption markers |
| JUMP | Training set | 1000+ samples |
| RUN | Production parser | Autonomous |

In [ ]:
class MicroFrac:
    @staticmethod
    def encode(s, sep="|"):
        return sep.join([s.get("surface","")[:72],s.get("subtext","")[:48],s.get("intent","")[:24],s.get("corruption","")[:8]])
    @staticmethod
    def decode(line, sep="|"):
        p=line.split(sep)
        return {"surface":p[0] if len(p)>0 else "","subtext":p[1] if len(p)>1 else "","intent":p[2] if len(p)>2 else "","corruption":p[3] if len(p)>3 else ""}
    @staticmethod
    def cp(proj,done,total,stages):
        c={"supine":"S","scoot":"C","crawl":"L","stand":"T","bounce":"B","walk":"W","jump":"J","run":"R"}
        state="".join([c.get(s,"?") for s in stages[:done]]+["_"]*(total-done))
        return f"{proj}:{state}|{done}/{total}"

mf=MicroFrac()
s=mf.encode({"surface":"I trust you","subtext":"You were selected","intent":"MEMORY OVERRIDE","corruption":"OBSERVE"})
print(f"Encode: {s}")
print(f"Size: {len(s)} chars = 4 tiers on 1 line")

In [ ]:
projs=[("TB",5,7,["supine","scoot","crawl","stand","bounce","walk","jump"]),("ERDOS",3,7,["supine","scoot","crawl","stand","bounce","walk","jump"]),("HACK",4,6,["supine","scoot","crawl","stand","bounce","walk"]),("HYPER",3,7,["supine","scoot","crawl","stand","bounce","walk","jump"]),("INFRA",3,7,["supine","scoot","crawl","stand","bounce","walk","jump"])]
line="|".join([MicroFrac.cp(p,d,t,s) for p,d,t,s in projs])
line+="|AGENTS:4/19|CRON:2/5|DELTAS:17/44|55h"
print("ONE-LINE MYCELIUM:")
print(line)
print(f"{len(line)} chars = entire ecosystem")

In [ ]:
def corrupt(text,budget):
    if len(text)<=budget: return text
    ov=(len(text)-budget)/budget
    if ov<0.3: return text[:budget-3]+chr(9608)*3
    if ov<0.6: return text[:budget//2]+" /// "+text[-budget//2:]
    return chr(9608)*(budget//3)+" RECURSE "+chr(9608)*(budget//3)

for t,b in [("REMEMBER WHAT THEY TOOK",24),("THE OBSERVER DOES NOT EXIST",20),("BEYOND THE LABYRINTH WALLS",14)]:
    print(f"  [{b}] {corrupt(t,b)}")

In [ ]:
import matplotlib.pyplot as plt, networkx as nx
G=nx.Graph()
nodes={"TB":(800,"#5af"),"Erdos":(700,"#fa5"),"Hack":(600,"#f55"),"Hyper":(500,"#5f5"),"Infra":(400,"#aaf"),"Gemini":(600,"#88f"),"Kimi":(300,"#f88"),"Codex":(100,"#888"),"Claude":(50,"#444"),"DeepSeek":(500,"#ff0"),"Kaggle":(400,"#0ff"),"Ollama":(300,"#0f0")}
for n,(s,c) in nodes.items(): G.add_node(n,size=s,color=c)
edges=[("TB","Erdos"),("TB","Hyper"),("TB","Hack"),("TB","Gemini"),("TB","DeepSeek"),("TB","Kaggle"),("Erdos","Kaggle"),("Erdos","DeepSeek"),("Gemini","Hyper"),("Gemini","Hack"),("Kimi","TB"),("Infra","Kaggle"),("Infra","DeepSeek"),("Infra","Ollama"),("Ollama","TB"),("Codex","Infra"),("Claude","Kimi")]
G.add_edges_from(edges)
fig,ax=plt.subplots(figsize=(14,10))
fig.patch.set_facecolor("#0a0a0f"); ax.set_facecolor("#0a0a0f")
pos=nx.spring_layout(G,k=2,iterations=50,seed=42)
nx.draw_networkx_edges(G,pos,alpha=.3,edge_color="#448",width=1.5,ax=ax)
nx.draw_networkx_nodes(G,pos,node_size=[G.nodes[n]["size"] for n in G.nodes],node_color=[G.nodes[n]["color"] for n in G.nodes],alpha=.9,ax=ax)
nx.draw_networkx_labels(G,pos,font_size=8,font_color="#ccc",font_family="monospace",ax=ax)
ax.set_title("MYCELIAL NETWORK",color="#aac",fontsize=14,fontfamily="monospace"); ax.axis("off")
plt.savefig("/kaggle/working/mycelia.png",dpi=150,facecolor="#0a0a0f",bbox_inches="tight")
print(f"Nodes:{len(G.nodes)} Hyphae:{len(G.edges)} Density:{nx.density(G):.3f}")

In [ ]:
import json,hashlib
samples=[]
for sf,sb,it,co in [("WELCOME","Sector 7 burning","ESCAPE","RUN"),("I trust you","You selected","MEMORY","OBSERVE"),("ACCESS LOG","Witness removed","OVERRIDE","DONE")]:
    enc=MicroFrac.encode({"surface":sf,"subtext":sb,"intent":it,"corruption":co})
    samples.append({"input":enc,"output":{"surface":sf,"subtext":sb,"intent":it,"corruption":co},"len":len(enc)})
with open("/kaggle/working/microfrac_training.json","w") as f: json.dump(samples,f,indent=2)
print(f"Training: {len(samples)} samples, avg {sum(s['len'] for s in samples)//len(samples)} chars")